# 실습 2. 모델을 복잡하게 만들면 더 똑똑해질까?

## 학습 목표
이번 실습에서는 다항회귀 모델의 복잡도를 바꾸어 보면서 다음 내용을 확인한다.

- 훈련 데이터와 테스트 데이터를 구분하는 이유를 이해한다.
- 모델의 복잡도에 따라 과소적합과 과적합이 발생하는 것을 확인한다.
- 훈련 MSE와 테스트 MSE를 비교하여 일반화 성능을 판단한다.
- 생성형 AI를 이용하여 실험 결과를 해석하고 다음 실험을 설계한다.

> 핵심 질문: **훈련 데이터에서 가장 잘 맞는 모델이 정말 가장 좋은 모델일까?**

## 1. 실험용 데이터 만들기

실제 데이터에는 일정한 규칙과 함께 우연한 잡음(noise)이 섞여 있다.
아래에서는 이러한 상황을 간단한 인공 데이터로 만들어 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

X = np.linspace(-3, 3, 30)
y = 0.5 * X**3 - 2 * X + np.random.normal(0, 3, 30)

plt.figure(figsize=(7, 4))
plt.scatter(X, y)
plt.xlabel("X")
plt.ylabel("y")
plt.title("Observed data")
plt.show()

### 생각해 보기
점들의 전체적인 흐름을 잘 설명하려면 어떤 모양의 선이 적절할까?

- 직선이면 충분할까?
- 곡선이 필요할까?
- 모든 점을 정확히 지나가는 선이 가장 좋은 모델일까?

## 2. 훈련 데이터와 테스트 데이터로 나누기

모델은 훈련 데이터로 학습한다.  
테스트 데이터는 학습에 사용하지 않고, 학습이 끝난 모델의 **새로운 데이터에 대한 성능**을 평가하는 데 사용한다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print("훈련 데이터 개수:", len(X_train))
print("테스트 데이터 개수:", len(X_test))

## 3. 다항회귀 모델 학습

다항회귀에서 `degree`는 모델의 복잡도를 조절한다.

- 작은 `degree` → 단순한 모델
- 큰 `degree` → 복잡한 모델

먼저 `degree=1`부터 시작해 보자.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

degree = 1

model = make_pipeline(
    PolynomialFeatures(degree),
    LinearRegression()
)

model.fit(X_train.reshape(-1, 1), y_train)

## 4. 학습된 모델을 그림으로 확인하기

In [ ]:
X_grid = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_grid = model.predict(X_grid)

plt.figure(figsize=(7, 4))
plt.scatter(X_train, y_train, label="Train")
plt.scatter(X_test, y_test, marker="x", s=70, label="Test")
plt.plot(X_grid, y_grid, linewidth=2, label=f"degree={degree}")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Polynomial regression")
plt.legend()
plt.show()

### 실험
위의 `degree` 값을 다음과 같이 바꾸어 실행해 본다.

1. `degree = 1`
2. `degree = 3`
3. `degree = 15`

각 모델의 곡선 모양을 비교해 보자.

- 어떤 모델이 너무 단순한가?
- 어떤 모델이 데이터의 전체적인 흐름을 잘 설명하는가?
- 어떤 모델이 훈련 데이터의 작은 변화까지 지나치게 따라가는가?

## 5. MSE로 성능 측정하기

그림만 보고 판단하지 않고, 평균제곱오차(MSE)를 이용하여 성능을 숫자로 확인한다.

In [ ]:
from sklearn.metrics import mean_squared_error

train_pred = model.predict(X_train.reshape(-1, 1))
test_pred = model.predict(X_test.reshape(-1, 1))

train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)

print(f"degree = {degree}")
print(f"Train MSE = {train_mse:.3f}")
print(f"Test  MSE = {test_mse:.3f}")

### 결과 해석
일반적으로 다음과 같은 현상을 관찰할 수 있다.

- **과소적합**: Train MSE도 크고 Test MSE도 크다.
- **적절한 학습**: Train MSE와 Test MSE가 모두 비교적 작다.
- **과적합**: Train MSE는 매우 작지만 Test MSE가 상대적으로 커진다.

> 중요한 점: **훈련 오차가 작다는 사실만으로 좋은 모델이라고 할 수 없다.**

## 6. 모델 복잡도를 1차부터 15차까지 비교하기

이제 모델의 차수를 하나씩 바꾸면서 훈련 MSE와 테스트 MSE가 어떻게 변하는지 확인한다.

In [ ]:
degrees = range(1, 16)
train_mse_list = []
test_mse_list = []

for d in degrees:
    model = make_pipeline(
        PolynomialFeatures(d),
        LinearRegression()
    )
    model.fit(X_train.reshape(-1, 1), y_train)

    train_pred = model.predict(X_train.reshape(-1, 1))
    test_pred = model.predict(X_test.reshape(-1, 1))

    train_mse_list.append(mean_squared_error(y_train, train_pred))
    test_mse_list.append(mean_squared_error(y_test, test_pred))

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_mse_list, marker="o", label="Train MSE")
plt.plot(degrees, test_mse_list, marker="o", label="Test MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("MSE")
plt.title("Model complexity and generalization")
plt.xticks(list(degrees))
plt.legend()
plt.show()

### 핵심 관찰
그래프를 보고 다음 질문에 답해 보자.

1. `degree`가 증가할수록 Train MSE는 어떻게 변하는가?
2. Test MSE가 가장 작은 차수는 어느 정도인가?
3. Train MSE와 Test MSE의 차이가 커지는 구간은 어디인가?
4. 그 구간을 **과적합**이라는 용어로 설명할 수 있는가?

## 7. 생성형 AI를 실험 조교로 활용하기

생성형 AI에게 단순히 "코드를 작성해 줘"라고 요청하기보다,
**결과의 이유를 설명하고 다음 실험을 제안하도록** 활용해 보자.

### 프롬프트 1: 결과 해석
다음 문장을 생성형 AI에 입력해 본다.

> 다항회귀에서 degree를 1, 3, 15로 바꾸어 실험했다.  
> degree=15에서 Train MSE는 매우 작지만 Test MSE는 더 커졌다.  
> 이 현상을 과소적합, 과적합, 일반화라는 용어를 사용하여 설명해 줘.

### 프롬프트 2: 다음 실험 설계
> 과적합인지 더 확실하게 확인하려면 어떤 추가 실험을 해보면 좋을까?  
> 코드를 바로 작성하지 말고 실험 방법을 먼저 제안해 줘.

## 8. 마무리

이번 실습에서 확인한 핵심은 다음과 같다.

- 모델은 훈련 데이터에서 단순히 정답을 외우는 것이 목적이 아니다.
- 모델이 복잡해지면 훈련 데이터에는 더 잘 맞을 수 있다.
- 그러나 지나치게 복잡하면 새로운 데이터에 대한 성능은 오히려 나빠질 수 있다.
- 따라서 모델의 진짜 실력은 **보지 않은 데이터에 대한 성능**, 즉 일반화 성능으로 평가해야 한다.

### 마지막 질문
**훈련 데이터에서 가장 정확한 모델이 항상 가장 좋은 AI 모델일까?**